# Fresh Start NBA — Notebook 3: Feature Engineering

This notebook mirrors the `feature_pipeline.py` logic, broken into visible stages so you can
inspect intermediate outputs at each step and experiment with new features.

**Stages:**
1. Base rolling averages (L5, L10, L20)
2. EWMA-weighted features
3. Opponent context (defense strength, pace)
4. Situational splits (home/away, B2B, defense tier)
5. Usage & role trends
6. Synthetic lines (Vegas intelligence)
7. Export feature-engineered dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR   = Path('/content/drive/MyDrive/Fresh_Start_NBA_Colab')
DATA_DIR   = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
OUT_DIR    = BASE_DIR / 'outputs'

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 110

nba = pd.read_csv(DATA_DIR / 'nba_data.csv', low_memory=False)
nba['game_date'] = pd.to_datetime(nba['game_date'])
nba = nba.sort_values(['player', 'game_date']).reset_index(drop=True)
print(f'Loaded: {len(nba):,} rows, {nba["player"].nunique()} unique players')

## Stage 1 — Base Rolling Averages (L5, L10, L20)

In [ ]:
STAT_COLS = ['pts', 'trb', 'ast', 'stl', 'blk', 'tov', 'mp', 'fga', 'fta', '3pa', 'fg_pct', 'ft_pct', '3p_pct']
WINDOWS = [5, 10, 20]

df = nba.copy()

for stat in STAT_COLS:
    if stat not in df.columns:
        continue
    grp = df.groupby('player')[stat]
    for w in WINDOWS:
        df[f'{stat}_l{w}'] = grp.transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
    # Std dev
    df[f'{stat}_std_l10'] = grp.transform(lambda x: x.shift(1).rolling(10, min_periods=3).std())

# Days rest and B2B flags
df['days_rest'] = df.groupby('player')['game_date'].diff().dt.days
df['is_b2b'] = (df['days_rest'] == 1).astype(int)
df['is_b2b_second'] = df['is_b2b']   # same concept

# Cumulative games played
df['games_played'] = df.groupby('player').cumcount() + 1

new_rolling_cols = [c for c in df.columns if '_l5' in c or '_l10' in c or '_l20' in c]
print(f'Stage 1 complete. Added {len(new_rolling_cols)} rolling feature columns.')
df[['player', 'game_date', 'pts', 'pts_l5', 'pts_l10', 'pts_l20', 'pts_std_l10']].tail(8)

## Stage 2 — EWMA-Weighted Features

In [ ]:
EWMA_SPAN = 5  # tune this: smaller = faster decay, larger = smoother

EWMA_STATS = ['pts', 'trb', 'ast', 'mp', 'fga', 'fta', '3pa', 'stl', 'blk', 'tov']

for stat in EWMA_STATS:
    if stat not in df.columns:
        continue
    # Shift 1 to avoid lookahead, then EWMA
    df[f'{stat}_ewma{EWMA_SPAN}'] = (
        df.groupby('player')[stat]
        .transform(lambda x: x.shift(1).ewm(span=EWMA_SPAN, min_periods=1).mean())
    )
    # EWMA std and consistency score
    df[f'{stat}_ewma_std'] = (
        df.groupby('player')[stat]
        .transform(lambda x: x.shift(1).ewm(span=EWMA_SPAN, min_periods=3).std())
    )
    df[f'{stat}_consistency'] = (
        df[f'{stat}_ewma{EWMA_SPAN}'] /
        (df[f'{stat}_ewma_std'] + 1e-6)
    )

ewma_cols = [c for c in df.columns if 'ewma' in c or 'consistency' in c]
print(f'Stage 2 complete. Added {len(ewma_cols)} EWMA columns.')
df[['player', 'game_date', 'pts', 'pts_ewma5', 'pts_ewma_std', 'pts_consistency']].tail(5)

## Stage 3 — Opponent Context (Defense Strength, Pace)

In [ ]:
# Extract opponent from matchup string (e.g. 'LAL vs. BOS' or 'LAL @ BOS')
if 'team' not in df.columns and 'TEAM_ABBREVIATION' in df.columns:
    df['team'] = df['TEAM_ABBREVIATION']

def extract_opponent(row):
    matchup = str(row.get('matchup', ''))
    team = str(row.get('team', ''))
    parts = matchup.replace('@', 'vs.').split('vs.')
    if len(parts) == 2:
        for p in parts:
            p = p.strip()
            if p != team:
                return p
    return None

df['opponent'] = df.apply(extract_opponent, axis=1)
df['is_home'] = df['matchup'].str.contains('vs.').astype(int)

# Rolling opponent defensive stats (pts, trb, ast allowed per game)
for stat in ['pts', 'trb', 'ast']:
    opp_avg = (
        df.groupby(['opponent', 'game_date'])[stat].mean()
        .reset_index()
        .rename(columns={stat: f'_opp_game_{stat}'})
    )
    opp_avg = opp_avg.sort_values('game_date')
    opp_avg[f'opp_{stat}_allowed_l10'] = (
        opp_avg.groupby('opponent')[f'_opp_game_{stat}']
        .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
    )
    opp_avg[f'opp_{stat}_allowed_rank'] = (
        opp_avg.groupby('game_date')[f'opp_{stat}_allowed_l10']
        .transform(lambda x: x.rank(pct=True))
    )
    df = df.merge(
        opp_avg[['opponent', 'game_date', f'opp_{stat}_allowed_l10', f'opp_{stat}_allowed_rank']],
        on=['opponent', 'game_date'], how='left'
    )

# Pace proxy: average FGA allowed by opponent
opp_pace = (
    df.groupby(['opponent', 'game_date'])['fga'].mean()
    .reset_index().rename(columns={'fga': '_opp_fga'})
    .sort_values('game_date')
)
opp_pace['opp_pace_factor'] = (
    opp_pace.groupby('opponent')['_opp_fga']
    .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
)
df = df.merge(opp_pace[['opponent', 'game_date', 'opp_pace_factor']], on=['opponent', 'game_date'], how='left')

# Defense tier (0=best, 2=worst)
df['opp_def_tier'] = pd.cut(df['opp_pts_allowed_rank'], bins=[0, 0.33, 0.67, 1.0], labels=[0, 1, 2]).astype(float)

opp_cols = [c for c in df.columns if c.startswith('opp_')]
print(f'Stage 3 complete. Added {len(opp_cols)} opponent context columns.')
df[['player', 'game_date', 'opponent', 'opp_pts_allowed_l10', 'opp_pts_allowed_rank', 'opp_def_tier']].tail(5)

## Stage 4 — Situational Splits

In [ ]:
SPLIT_WINDOW = 15

for stat in ['pts', 'trb', 'ast', 'mp']:
    if stat not in df.columns:
        continue

    # Home vs away splits
    for home_flag, suffix in [(1, 'home'), (0, 'away')]:
        mask = df['is_home'] == home_flag
        tmp = df[mask].groupby('player')[stat]
        df.loc[mask, f'{stat}_{suffix}_l{SPLIT_WINDOW}'] = tmp.transform(
            lambda x: x.shift(1).rolling(SPLIT_WINDOW, min_periods=3).mean()
        )

    df[f'{stat}_home_l{SPLIT_WINDOW}'] = df[f'{stat}_home_l{SPLIT_WINDOW}'].fillna(df[f'{stat}_l{SPLIT_WINDOW}'])
    df[f'{stat}_away_l{SPLIT_WINDOW}'] = df[f'{stat}_away_l{SPLIT_WINDOW}'].fillna(df[f'{stat}_l{SPLIT_WINDOW}'])
    df[f'{stat}_home_away_diff'] = df[f'{stat}_home_l{SPLIT_WINDOW}'] - df[f'{stat}_away_l{SPLIT_WINDOW}']
    df[f'{stat}_situational_avg'] = np.where(
        df['is_home'] == 1, df[f'{stat}_home_l{SPLIT_WINDOW}'], df[f'{stat}_away_l{SPLIT_WINDOW}']
    )

    # B2B penalty
    b2b_mask = df['is_b2b'] == 1
    df.loc[b2b_mask, f'{stat}_on_b2b'] = df.loc[b2b_mask].groupby('player')[stat].transform(
        lambda x: x.shift(1).rolling(10, min_periods=2).mean()
    )
    df[f'{stat}_rested'] = df[f'{stat}_l{SPLIT_WINDOW}']
    df[f'{stat}_on_b2b'] = df[f'{stat}_on_b2b'].fillna(df[f'{stat}_l{SPLIT_WINDOW}'])
    df[f'{stat}_b2b_penalty'] = df[f'{stat}_on_b2b'] - df[f'{stat}_rested']

print('Stage 4 complete.')
df[['player', 'game_date', 'is_home', 'pts_home_l15', 'pts_away_l15', 'pts_home_away_diff', 'pts_b2b_penalty']].tail(5)

## Stage 5 — Usage & Role Trends

In [ ]:
# Usage proxy: fga / team_fga
team_fga = df.groupby(['team', 'game_date'])['fga'].transform('sum')
df['usage_proxy'] = df['fga'] / (team_fga + 1e-6)

grp = df.groupby('player')
df['usage_l5']  = grp['usage_proxy'].transform(lambda x: x.shift(1).rolling(5,  min_periods=2).mean())
df['usage_l10'] = grp['usage_proxy'].transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
df['usage_trend'] = df['usage_l5'] - df['usage_l10']

# Role flags
df['mp_l3'] = grp['mp'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
df['mp_trend_ratio'] = df['mp_l3'] / (df['mp_l10'] + 1e-6)
df['role_expanding'] = (df['mp_trend_ratio'] > 1.10).astype(int)
df['role_shrinking'] = (df['mp_trend_ratio'] < 0.90).astype(int)
df['likely_starter'] = (df['mp_l5'] >= 25).astype(int)
df['high_min_prev']  = (df['mp_l3'] >= 30).astype(int)

# FGA trend
df['fga_l3'] = grp['fga'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
df['fga_trend'] = df['fga_l3'] - df['fga_l10']

# Trend slopes (L5 vs L20)
for stat in ['pts', 'trb', 'ast']:
    df[f'{stat}_trend'] = df[f'{stat}_l5'] - df[f'{stat}_l20']

print('Stage 5 complete.')
df[['player', 'game_date', 'usage_proxy', 'usage_l5', 'usage_trend', 'role_expanding', 'likely_starter']].tail(5)

## Stage 6 — Synthetic Vegas Lines

In [ ]:
# If real PrizePicks lines exist, merge them; otherwise use rolling averages as proxy
try:
    lines = pd.read_csv(DATA_DIR / 'historical_lines.csv', low_memory=False)
    lines['game_date'] = pd.to_datetime(lines['game_date'])

    PROP_TO_STAT = {
        'player_points': 'pts', 'player_rebounds': 'trb', 'player_assists': 'ast',
        'player_points_rebounds_assists': 'pra', 'player_points_rebounds': 'pr',
        'player_points_assists': 'pa', 'player_steals': 'stl',
        'player_blocks': 'blk', 'player_turnovers': 'tov'
    }

    for prop, stat in PROP_TO_STAT.items():
        prop_lines = lines[lines['prop'] == prop][['game_date', 'player', 'line']].copy()
        prop_lines = prop_lines.rename(columns={'line': f'{stat}_real_line'})
        df = df.merge(prop_lines, on=['game_date', 'player'], how='left')

    print('Real PrizePicks lines merged.')
except Exception as e:
    print(f'Could not merge real lines: {e}')

# Synthetic lines = EWMA (best estimate when no real line exists)
for stat in ['pts', 'trb', 'ast']:
    df[f'{stat}_synthetic_line'] = df[f'{stat}_ewma5']
    df[f'{stat}_edge_vs_avg'] = df[f'{stat}_l10'] - df[f'{stat}_synthetic_line']
    df[f'{stat}_vegas_slow'] = df[f'{stat}_l20']  # "stale" line proxy
    df[f'{stat}_pct_diff'] = (
        (df[f'{stat}_synthetic_line'] - df[f'{stat}_vegas_slow']) /
        (df[f'{stat}_vegas_slow'].abs() + 1e-6)
    )
    df[f'{stat}_best_estimate'] = df[f'{stat}_ewma5']
    df[f'{stat}_confidence'] = 1.0 / (df[f'{stat}_std_l10'] + 1.0)

print('Stage 6 complete.')
df[['player', 'game_date', 'pts_synthetic_line', 'pts_edge_vs_avg', 'pts_confidence']].tail(5)

## Stage 7 — Validate & Export

In [ ]:
# Load the official feature column list
with open(BASE_DIR / 'feature_cols_advanced.json') as f:
    feat_meta = json.load(f)
FEATURE_COLS = feat_meta['feature_columns']

available = [c for c in FEATURE_COLS if c in df.columns]
missing   = [c for c in FEATURE_COLS if c not in df.columns]

print(f'Feature coverage: {len(available)}/{len(FEATURE_COLS)} columns present')
if missing:
    print(f'Missing {len(missing)} columns (require full pipeline or advanced model features):')
    print(missing[:20])

In [ ]:
# Export the feature-engineered dataset
out_path = OUT_DIR / 'nba_features.csv'
df.to_csv(out_path, index=False)
print(f'Saved feature-engineered dataset: {out_path}')
print(f'Shape: {df.shape}')

## Adding a New Feature

To add a new feature, add a new cell here and compute it on `df`, then add its name to `FEATURE_COLS` above and re-run notebooks 4 and 5 to see if it improves model accuracy.

In [ ]:
# EXPERIMENT CELL — add new features here
#
# Example: pts per minute as an efficiency proxy
# df['pts_per_min'] = df['pts_l10'] / (df['mp_l10'] + 1e-6)
#
# Example: opponent-adjusted pts (how much better/worse than opp allows)
# df['pts_opp_adj'] = df['pts_l10'] / (df['opp_pts_allowed_l10'] + 1e-6)

pass